## Script Review of Daily Gas Burn By Site

Notebook is for reviewing Clarissa's Daily Gas Burn Script. I have a copy of the version from 7/17/2026 saved within this project ('gas burn forecasting model (7-09 Update) - Copy.py). Notebook was created by copy and pasting from the original script. Additional code added by me will be called out withing the blocks.

Notes:
- I don't yet have access to the Allegro database

In [1]:
#importing the libraries needed
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from datetime import timedelta
import os
from pathlib import Path
from sklearn.metrics import mean_squared_error
from mssql_python import connect
from nbdevAuto.functions import * 
import nbdevAuto.functions
import time 

### Environment Variables

loading from the .env file

In [2]:
#to make sure the variables for the api are on the local os  
YES_USER = os.getenv("YES_USER")
YES_PASS = os.getenv("YES_PASS")

print("YES_USER loaded:", YES_USER is not None)
print("YES_PASS loaded:", YES_PASS is not None)

YES_USER loaded: False
YES_PASS loaded: False


### Loading Allegro Data

I don't have Allegro DB access yet, so I will just be loading the CSVs that Clarissa saved to 

In [ ]:
# DB connection
SQL_CONNECTION_STRING = ( "Server=HDQv1958;" "Database=allegro;" "Trusted_Connection=yes;" "Encrypt=yes;" "TrustServerCertificate=yes;")

conn = connect(SQL_CONNECTION_STRING)

In [ ]:
# DO NOT RUN
def load_burn():

    query = """
SELECT t.trade, p.marketarea, q.begtime, q.energy, q.quantitystatus

FROM trade t, position p, ngquantity q

WHERE p.bepc_strategy = 'Burn' and t.trade = p.trade and t.tradestatus <> 'Void' and p.position = q.position and q.posstatus = 1
AND q.begtime >= DATEADD(year, -2, GETDATE()) AND q.begtime <= GETDATE()

ORDER BY q.begtime"""

    df = pd.read_sql(query, conn)
    #df["datetime"] = pd.to_datetime(df["datetime"])

    return df[["begtime", "marketarea", "energy"]]

gas_daily_df = load_burn()

# DO NOT RUN

#defining market areas as sites
gas_daily_df["marketarea"] = gas_daily_df["marketarea"].str.upper().str.strip()

gas_daily_df = gas_daily_df[
    ~gas_daily_df["marketarea"].isin(["BISON", "COTTAGE GROVE"])]

site_map = {"DEER CREEK": "DCS", "LANARK": "CGS", "LONSOME CREEK": "LCS",  "STATELINE": "PGS", "GROTON": "GGS", "CULBERTSON": "CGS"}

gas_daily_df["site"] = gas_daily_df["marketarea"].replace(site_map)

# create gas day
gas_daily_df["gas_day"] = pd.to_datetime(gas_daily_df["begtime"])
gas_daily_df["gas_day"] = (gas_daily_df["gas_day"] - pd.Timedelta(hours=9)).dt.date

# aggregate on gas_day 
gas_daily_site = (gas_daily_df.groupby(["gas_day", "site"], as_index=False)["energy"].sum().rename(columns={"energy": "daily_gas_burn"}))

############### CheckPoint 1 (the above information) ###############
# gas_daily_site.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint1.csv", index=False)

In [4]:
# Dataframe isn't really used in later code.
gas_daily_df = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 1.csv")
print(gas_daily_df.shape)
print(gas_daily_df.head())

(4262, 3)
      begtime     marketarea   energy
0  2024-07-23      Stateline  33933.0
1  2024-07-23  Lonsome Creek  42282.0
2  2024-07-23         Lanark  13523.0
3  2024-07-23     Deer Creek  46474.0
4  2024-07-23         Groton  11036.0


In [5]:
#Adding the columns from the data that we need in the final dataset

#defining market areas as sites
gas_daily_df["marketarea"] = gas_daily_df["marketarea"].str.upper().str.strip()

gas_daily_df = gas_daily_df[
    ~gas_daily_df["marketarea"].isin(["BISON", "COTTAGE GROVE"])]

site_map = {"DEER CREEK": "DCS", "LANARK": "CGS", "LONSOME CREEK": "LCS",  "STATELINE": "PGS", "GROTON": "GGS", "CULBERTSON": "CGS"}

gas_daily_df["site"] = gas_daily_df["marketarea"].replace(site_map)



# Caleb note: I changed this following portion to add a column titled gas_day, but it's just a renaming of the begtime column
# create gas day
gas_daily_df["gas_day"] = gas_daily_df['begtime']
#gas_daily_df["gas_day"] = (gas_daily_df["gas_day"] - pd.Timedelta(hours=9)).dt.date


# aggregate on gas_day and renaming energy column to daily_gas_burn 
gas_daily_site = (gas_daily_df.groupby(["gas_day", "site"], as_index=False)["energy"].sum().rename(columns={"energy": "daily_gas_burn"}))

#### Loading Non-PGS **Net Generation** Data



In [ ]:
# Do NOT RUN
def load_generation_nonpgs():

    query = """
    SELECT begtime, loadshape, he1, he2, he3, he4, he5, he6, he7, he8, he9, he10, he11, he12, he13, he14, he15, he16, he17, he18, he19, he20, he21, he22, he23, he24
    FROM dbo.loadshapeprofile
    WHERE begtime >= DATEADD(year, -3, GETDATE()) AND begtime <= GETDATE()
      AND loadshape IN ('WAUE.BEPM.DCS1 - Net Generation',
        'WAUE.BEPM.LCS1 - Net Generation', 'WAUE.BEPM.LCS2 - Net Generation', 'WAUE.BEPM.LCS3 - Net Generation', 'WAUE.BEPM.LCS4 - Net Generation', 'WAUE.BEPM.LCS5 - Net Generation',
        'WAUE.BEPM.LCS6 - Net Generation', 'WAUE.BEPM.GGS1 - Net Generation', 'WAUE.BEPM.GGS2 - Net Generation', 'WAUE.BEPM.CULBERTSON1 - Net Generation')
    """

    return pd.read_sql(query, conn)

df_nonpgs = load_generation_nonpgs()

############### CheckPoint 2 (the above information) ###############
#df_nonpgs.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint2.csv", index=False)

In [6]:
df_nonpgs = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 2.csv")
print(f'Rows and Columns of the non-PGS sites: {df_nonpgs.shape}')
df_nonpgs.head()

Rows and Columns of the non-PGS sites: (7300, 26)


,begtime,loadshape,he1,he2,he3,he4,he5,he6,he7,he8,...,he15,he16,he17,he18,he19,he20,he21,he22,he23,he24
0,2024-07-23,WAUE.BEPM.CULBERTSON1 - Net Generation,39.9,40.1,39.8,40.2,40.4,40.4,40.1,40.2,...,65.1,63.8,61.7,60.9,61.7,63.9,65.1,66.9,66.5,39.0
1,2024-07-24,WAUE.BEPM.CULBERTSON1 - Net Generation,38.0,38.1,37.9,38.1,38.0,38.3,38.3,38.3,...,58.0,56.3,55.3,54.8,54.7,56.1,58.3,62.7,62.2,42.2
2,2024-07-25,WAUE.BEPM.CULBERTSON1 - Net Generation,39.5,40.2,40.1,40.2,40.1,40.1,40.2,39.8,...,58.8,57.5,56.9,57.7,59.7,61.4,62.2,63.6,60.6,59.8
3,2024-07-26,WAUE.BEPM.CULBERTSON1 - Net Generation,42.6,46.8,47.2,44.1,38.1,38.1,38.1,38.2,...,59.9,60.8,60.4,61.4,62.9,63.6,61.2,60.6,61.3,60.9
4,2024-07-27,WAUE.BEPM.CULBERTSON1 - Net Generation,63.7,47.1,46.3,46.8,42.7,38.3,38.0,38.5,...,64.2,61.8,61.3,60.8,64.7,62.9,63.6,60.7,79.6,39.2


#### Loading PGS **Net Generation** Data

Note: 
- this data comes in at five minute intervals in Allegro. The SQL query provided by IT (Tyler Herman?) does the aggregation for us. It comes in by unit but aggregated by HE##
- It looks like only some of the units are 5 Minute Intervals
    - PGS 4, PGS 5 only show hourly in Allegro (unless I'm dumb). Is there something unique about these units?

In [ ]:
def load_generation_pgs():

    query = """SELECT begtime, loadshape, SUM(he1)  AS he1, SUM(he2)  AS he2, SUM(he3)  AS he3, SUM(he4)  AS he4, SUM(he5)  AS he5, SUM(he6)  AS he6, SUM(he7)  AS he7,
    SUM(he8)  AS he8, SUM(he9)  AS he9, SUM(he10) AS he10, SUM(he11) AS he11, SUM(he12) AS he12, SUM(he13) AS he13, SUM(he14) AS he14, SUM(he15) AS he15, SUM(he16) AS he16,
    SUM(he17) AS he17, SUM(he18) AS he18, SUM(he19) AS he19, SUM(he20) AS he20, SUM(he21) AS he21, SUM(he22) AS he22, SUM(he23) AS he23, SUM(he24) AS he24

    FROM dbo.loadshapeprofile

    WHERE loadshape LIKE '%PGS%' AND loadshape LIKE '%- Net Generation-5m' AND begtime >= DATEADD(year, -2, GETDATE()) AND begtime <= GETDATE()

    GROUP BY begtime, loadshape

    ORDER BY begtime;"""

    return pd.read_sql(query, conn)

df_pgs = load_generation_pgs()

############### CheckPoint 3 (the above information) ###############
#df_pgs.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint3.csv", index=False)

In [7]:
df_pgs = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 3.csv")
print(f'Rows and Columns of the non-PGS sites: {df_pgs.shape}')
df_pgs.head()

Rows and Columns of the non-PGS sites: (17426, 26)


,begtime,loadshape,he1,he2,he3,he4,he5,he6,he7,he8,...,he15,he16,he17,he18,he19,he20,he21,he22,he23,he24
0,2024-07-23,WAUE.BEPM.PGS14_16 - Net Generation-5m,17.9,17.9,18.0,17.9,17.9,17.9,17.9,17.7,...,17.9,17.9,17.8,17.9,17.9,17.9,17.9,17.9,16.8,12.4
1,2024-07-23,WAUE.BEPM.PGS17_19 - Net Generation-5m,17.8,17.8,17.9,17.9,17.8,17.9,17.8,17.9,...,17.9,10.6,9.0,8.9,8.9,8.9,9.0,8.9,8.8,7.8
2,2024-07-23,WAUE.BEPM.PGS13 - Net Generation-5m,9.0,8.9,8.8,8.9,9.0,8.9,9.0,8.8,...,8.9,8.9,8.9,8.9,9.0,7.4,7.5,7.4,6.9,4.4
3,2024-07-23,WAUE.BEPM.PGS11 - Net Generation-5m,8.9,8.9,9.0,8.9,8.9,9.0,8.9,9.0,...,9.0,8.9,9.0,8.8,9.0,8.9,9.0,8.9,8.2,4.6
4,2024-07-23,WAUE.BEPM.PGS2 - Net Generation-5m,39.6,39.5,39.5,39.3,39.6,39.6,39.5,39.6,...,38.8,38.8,38.8,38.5,38.8,38.4,38.6,38.4,36.0,31.4


### Transforming Allegro Data - Generation

In [8]:
#Combining the data sources into one dataframe and mapping multiple generators to their corisponding sites 
df_load_generation = pd.concat([df_nonpgs, df_pgs], ignore_index=True)

print(f'dimensions of df_load_generation: {df_load_generation.shape}')


dimensions of df_load_generation: (24726, 26)


In [ ]:
##################################### CHECKPOINT 4 ( the above information) ######################################################
#df_load_generation.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint4.csv", index=False)
df_load_generation.to_csv('../output-data/checkpoint4_test.csv', index=False)

df_load_generation.head()

In [9]:
df_load_generation.iloc[:, 2:].sum().sum()

np.float64(8166104.528999999)

In [10]:
df_load_generation.info()

<class 'pandas.DataFrame'>
RangeIndex: 24726 entries, 0 to 24725
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   begtime    24726 non-null  str    
 1   loadshape  24726 non-null  str    
 2   he1        24726 non-null  float64
 3   he2        24726 non-null  float64
 4   he3        24664 non-null  float64
 5   he4        24726 non-null  float64
 6   he5        24726 non-null  float64
 7   he6        24726 non-null  float64
 8   he7        24726 non-null  float64
 9   he8        24726 non-null  float64
 10  he9        24726 non-null  float64
 11  he10       24726 non-null  float64
 12  he11       24726 non-null  float64
 13  he12       24716 non-null  float64
 14  he13       24716 non-null  float64
 15  he14       24716 non-null  float64
 16  he15       24716 non-null  float64
 17  he16       24716 non-null  float64
 18  he17       24716 non-null  float64
 19  he18       24716 non-null  float64
 20  he19       24716 

In [11]:
# convert date using begtime
df_load_generation["begtime"] = pd.to_datetime(df_load_generation["begtime"])
df_load_generation["date"] = df_load_generation["begtime"].dt.date

In [12]:
# map site
df_load_generation["site"] = "N/A"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("DCS"), "site"] = "DCS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("LCS"), "site"] = "LCS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("PGS"), "site"] = "PGS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("GGS"), "site"] = "GGS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("CULBERTSON"), "site"] = "CGS"

##################################### CHECKPOINT 5 ( the above information) ######################################################
#df_load_generation.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint5.csv", index=False)

In [13]:
# pre pivoting/melting
df_load_generation.head()

,begtime,loadshape,he1,he2,he3,he4,he5,he6,he7,he8,...,he17,he18,he19,he20,he21,he22,he23,he24,date,site
0,2024-07-23,WAUE.BEPM.CULBERTSON1 - Net Generation,39.9,40.1,39.8,40.2,40.4,40.4,40.1,40.2,...,61.7,60.9,61.7,63.9,65.1,66.9,66.5,39.0,2024-07-23,CGS
1,2024-07-24,WAUE.BEPM.CULBERTSON1 - Net Generation,38.0,38.1,37.9,38.1,38.0,38.3,38.3,38.3,...,55.3,54.8,54.7,56.1,58.3,62.7,62.2,42.2,2024-07-24,CGS
2,2024-07-25,WAUE.BEPM.CULBERTSON1 - Net Generation,39.5,40.2,40.1,40.2,40.1,40.1,40.2,39.8,...,56.9,57.7,59.7,61.4,62.2,63.6,60.6,59.8,2024-07-25,CGS
3,2024-07-26,WAUE.BEPM.CULBERTSON1 - Net Generation,42.6,46.8,47.2,44.1,38.1,38.1,38.1,38.2,...,60.4,61.4,62.9,63.6,61.2,60.6,61.3,60.9,2024-07-26,CGS
4,2024-07-27,WAUE.BEPM.CULBERTSON1 - Net Generation,63.7,47.1,46.3,46.8,42.7,38.3,38.0,38.5,...,61.3,60.8,64.7,62.9,63.6,60.7,79.6,39.2,2024-07-27,CGS


In [14]:
# melt function instead of unpivot
# melt
hour_cols = [column for column in df_load_generation.columns if column.startswith("he")]

#hourly_df = df_load_generation.melt(id_vars=["begtime", "site"], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")

# Caleb proposed update to previous ste[] by adding loadshape. Goal is to make imputation more accurate/representative and keep original grain until aggregation becomes needed
hourly_df = df_load_generation.melt(id_vars=["begtime", "site", 'loadshape'], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")

#added by Caleb - Need to modify the hour column to properly sort
hourly_df['hour'] =  np.where(hourly_df['hour'].str.len()==3, hourly_df['hour'].str[:2] + '0' + hourly_df['hour'].str[2:], hourly_df['hour'])

# order rows logically by time
hourly_df = hourly_df.sort_values(by = ['site', 'loadshape', 'begtime', 'hour'], ascending=[False, False, True, True])

hourly_df.head()

,begtime,site,loadshape,hour,hourly_mw
12306,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0
37032,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0
61758,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0
86484,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0
111210,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0


In [15]:
hourly_df['hourly_mw'].sum() #still good

np.float64(8166104.528999999)

In [14]:
############### CheckPoint 6 (the above information) ###############
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint6.csv", index=False)
hourly_df.to_csv('../output-data/checkpoint6_test.csv', index=False)

In [16]:
#so not zero but not null?????????
#replace with previous value (use lag function)
# remove impossible generation spikes
#hourly_df.loc[hourly_df["hourly_mw"] > 2000, "hourly_mw"] = -9999 
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint7.csv", index=False)

#assign outliers to Nan prior to imputation
# DCS1 seems to have the highest repeated value at about 300. My assumption is anything higher than that is a system outlier
replaced_outlier = hourly_df["hourly_mw"] > 400
hourly_df.loc[hourly_df["hourly_mw"] > 400, "hourly_mw"] = np.nan # Nine records - 10501976.59 MW total

In [ ]:
hourly_df.to_csv('../output-data/checkpoint6_nan_check.csv', index=False)

In [17]:
print(hourly_df['hourly_mw'].sum()) # less because of the nine records that appear to be outliers
replaced_records = hourly_df['hourly_mw'].isna()
hourly_df[replaced_outlier]

7666944.435


,begtime,site,loadshape,hour,hourly_mw
303752,2025-11-05,LCS,WAUE.BEPM.LCS6 - Net Generation,he13,NaN
226956,2024-09-03,LCS,WAUE.BEPM.LCS3 - Net Generation,he10,NaN
520719,2024-08-05,GGS,WAUE.BEPM.GGS1 - Net Generation,he22,NaN
397579,2025-12-08,GGS,WAUE.BEPM.GGS1 - Net Generation,he17,NaN
175098,2026-01-30,GGS,WAUE.BEPM.GGS1 - Net Generation,he08,NaN
247457,2025-02-05,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he11,NaN


In [18]:
# Caleb Replacement thoughts
#   - If we keep grain by unit we can be a bit more precise on imputation/replacement
#   - Let's use ffill to replace the NaNs which have two sources
#       1. The daylight savings spring time shift (72 records)
#       2. Outlier removal (9 records)
hourly_df['hourly_mw'] = hourly_df['hourly_mw'].ffill()
hourly_df.to_csv('../output-data/checkpoint6_nan_filled_check.csv', index=False)

hourly_df[replaced_outlier] # 207.5 MW added back in

,begtime,site,loadshape,hour,hourly_mw
303752,2025-11-05,LCS,WAUE.BEPM.LCS6 - Net Generation,he13,0.0
226956,2024-09-03,LCS,WAUE.BEPM.LCS3 - Net Generation,he10,0.0
520719,2024-08-05,GGS,WAUE.BEPM.GGS1 - Net Generation,he22,82.9
397579,2025-12-08,GGS,WAUE.BEPM.GGS1 - Net Generation,he17,0.0
175098,2026-01-30,GGS,WAUE.BEPM.GGS1 - Net Generation,he08,0.0
247457,2025-02-05,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he11,0.0


In [19]:
hourly_df[replaced_outlier] # 207.5 MW added back in

,begtime,site,loadshape,hour,hourly_mw
303752,2025-11-05,LCS,WAUE.BEPM.LCS6 - Net Generation,he13,0.0
226956,2024-09-03,LCS,WAUE.BEPM.LCS3 - Net Generation,he10,0.0
520719,2024-08-05,GGS,WAUE.BEPM.GGS1 - Net Generation,he22,82.9
397579,2025-12-08,GGS,WAUE.BEPM.GGS1 - Net Generation,he17,0.0
175098,2026-01-30,GGS,WAUE.BEPM.GGS1 - Net Generation,he08,0.0
247457,2025-02-05,CGS,WAUE.BEPM.CULBERTSON1 - Net Generation,he11,0.0


In [20]:
hourly_df.head()

,begtime,site,loadshape,hour,hourly_mw
12306,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0
37032,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0
61758,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0
86484,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0
111210,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0


In [21]:
# extract hour_num 
hourly_df["hour_num"] = hourly_df["hour"].str.extract(r"(\d+)").astype(int)
hourly_df.head(24) # Caleb Check

,begtime,site,loadshape,hour,hourly_mw,hour_num
12306,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0,1
37032,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0,2
61758,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0,3
86484,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0,4
111210,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0,5
135936,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he06,0.0,6
160662,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he07,0.0,7
185388,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he08,0.0,8
210114,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he09,0.0,9
234840,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he10,0.0,10


In [22]:
# build datetime
hourly_df["datetime"] = (pd.to_datetime(hourly_df["begtime"]) + pd.to_timedelta(hourly_df["hour_num"] - 0, unit="h"))
print(f'shape: {hourly_df.shape}') #679224, 7
print(f'total MW: {hourly_df['hourly_mw'].sum()}')
hourly_df.head(24) # Caleb Check


shape: (593424, 7)
total MW: 7675861.152


,begtime,site,loadshape,hour,hourly_mw,hour_num,datetime
12306,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0,1,2025-04-01 01:00:00
37032,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0,2,2025-04-01 02:00:00
61758,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0,3,2025-04-01 03:00:00
86484,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0,4,2025-04-01 04:00:00
111210,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0,5,2025-04-01 05:00:00
135936,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he06,0.0,6,2025-04-01 06:00:00
160662,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he07,0.0,7,2025-04-01 07:00:00
185388,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he08,0.0,8,2025-04-01 08:00:00
210114,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he09,0.0,9,2025-04-01 09:00:00
234840,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he10,0.0,10,2025-04-01 10:00:00


In [23]:
#collapse duplicates   # check if duplicates exist
# this is really just aggregation at this point.
# We might be better off not doing this aggregation until later as we really just need to resort
hourly_df = (hourly_df.groupby(["datetime", "site", 'loadshape'], as_index=False)["hourly_mw"].sum())
hourly_df = hourly_df.sort_values(by = ['site', 'loadshape', 'datetime'], ascending=[False, False, True])

#Same MW as previous step should be good
# if we add loadshape to the groupby all shape and MWs are the same => Duplicates don't exist
print(f'Shape: {hourly_df.shape}')
print(f'total MW: {hourly_df['hourly_mw'].sum()}')
hourly_df.head(24) # Caleb Check

Shape: (593424, 4)
total MW: 7675861.152


,datetime,site,loadshape,hourly_mw
180467,2025-04-01 01:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180503,2025-04-01 02:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180539,2025-04-01 03:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180575,2025-04-01 04:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180611,2025-04-01 05:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180647,2025-04-01 06:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180683,2025-04-01 07:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180719,2025-04-01 08:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180755,2025-04-01 09:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180791,2025-04-01 10:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0


In [25]:
############### CheckPoint 7 (the above information) ###############
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint7.csv", index=False)
hourly_df.to_csv('../output-data/checkpoint7.csv', index=False)

In [25]:
##check this - should be good
# assign gas_day 
hourly_df["gas_day"] = (pd.to_datetime(hourly_df["datetime"]) - pd.Timedelta(hours=9)).dt.date

# extract hour #
hourly_df["hour"] = hourly_df["datetime"].dt.hour

hourly_df.head(24) # Caleb Check


,datetime,site,loadshape,hourly_mw,gas_day,hour
180467,2025-04-01 01:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,1
180503,2025-04-01 02:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,2
180539,2025-04-01 03:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,3
180575,2025-04-01 04:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,4
180611,2025-04-01 05:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,5
180647,2025-04-01 06:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,6
180683,2025-04-01 07:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,7
180719,2025-04-01 08:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,8
180755,2025-04-01 09:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-04-01,9
180791,2025-04-01 10:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-04-01,10


In [ ]:
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint8.csv", index=False)
hourly_df.to_csv('../output-data/checkpoint8.csv', index=False)

In [26]:
hourly_df.head(24)

,datetime,site,loadshape,hourly_mw,gas_day,hour
180467,2025-04-01 01:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,1
180503,2025-04-01 02:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,2
180539,2025-04-01 03:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,3
180575,2025-04-01 04:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,4
180611,2025-04-01 05:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,5
180647,2025-04-01 06:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,6
180683,2025-04-01 07:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,7
180719,2025-04-01 08:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-03-31,8
180755,2025-04-01 09:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-04-01,9
180791,2025-04-01 10:00:00,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0,2025-04-01,10


In [ ]:
#Caleb Note: SKIP - DO NOT RUN - step not needed

# build gas_day grid
gas_days = hourly_df["gas_day"].unique()
sites = hourly_df["site"].unique()
hours = np.arange(24)



full_grid = pd.MultiIndex.from_product( [gas_days, sites, hours], names=["gas_day", "site", "hour"])

#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint9.csv", index=False)

In [27]:
#this is where the code breaks down 
# apply grid

# Caleb Note: This is just reordering the columns. I added loadshape back in
hourly_df = hourly_df[["datetime","gas_day", "hour", "site", 'loadshape', "hourly_mw"]]

In [ ]:
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint10.csv", index=False)
hourly_df.to_csv('../output-data/checkpoint10.csv', index=False)

In [28]:
hourly_df.head()

,datetime,gas_day,hour,site,loadshape,hourly_mw
180467,2025-04-01 01:00:00,2025-03-31,1,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180503,2025-04-01 02:00:00,2025-03-31,2,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180539,2025-04-01 03:00:00,2025-03-31,3,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180575,2025-04-01 04:00:00,2025-03-31,4,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0
180611,2025-04-01 05:00:00,2025-03-31,5,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,0.0


In [29]:
hourly_generation_by_site = hourly_df.groupby(['datetime', 'gas_day', 'site']).agg({'hourly_mw': 'sum'}).reset_index().rename(columns={'hourly_mw': 'hourly_generation_mw'})
hourly_generation_by_site = hourly_generation_by_site.sort_values(by = ['site', 'datetime'])

In [ ]:
hourly_generation_by_site.to_csv('../output-data/checkpoint11 - Hourly Generation by Site.csv', index=False)


In [31]:
print(f'hourly generation: {hourly_generation_by_site['hourly_generation_mw'].sum()}')
hourly_generation_by_site.head()

hourly generation: 7675861.152


,datetime,gas_day,site,hourly_generation_mw
0,2024-07-23 01:00:00,2024-07-22,CGS,39.9
5,2024-07-23 02:00:00,2024-07-22,CGS,40.1
10,2024-07-23 03:00:00,2024-07-22,CGS,39.8
15,2024-07-23 04:00:00,2024-07-22,CGS,40.2
20,2024-07-23 05:00:00,2024-07-22,CGS,40.4


In [33]:
daily_generation_by_site = hourly_df.groupby(['gas_day', 'site']).agg({'hourly_mw': 'sum'}).reset_index().rename(columns={'hourly_mw': 'daily_generation_mw'})
daily_generation_by_site = daily_generation_by_site.sort_values(by = ['site', 'gas_day'])


In [ ]:
daily_generation_by_site.to_csv('../output-data/checkpoint11 - Daily Generation by Site.csv', index=False)


In [34]:
print(f'daily generation: {daily_generation_by_site['daily_generation_mw'].sum()}')
daily_generation_by_site.head()

daily generation: 7675861.151999999


,gas_day,site,daily_generation_mw
0,2024-07-22,CGS,321.1
5,2024-07-23,CGS,1273.9
10,2024-07-24,CGS,1215.8
15,2024-07-25,CGS,1253.8
20,2024-07-26,CGS,1292.2


In [35]:
#Hourly total by site
######################################################################################################################
hourly_site_gen_df = (hourly_df.groupby(["datetime", "gas_day", "hour", "site"], as_index=False)["hourly_mw"].sum().rename(columns={"hourly_mw": "hourly_site_gen_mw"}))
hourly_site_gen_df['gas_day'] = pd.to_datetime(hourly_site_gen_df['gas_day'])

print(hourly_site_gen_df.info())
hourly_site_gen_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            87600 non-null  datetime64[us]
 1   gas_day             87600 non-null  datetime64[s] 
 2   hour                87600 non-null  int32         
 3   site                87600 non-null  str           
 4   hourly_site_gen_mw  87600 non-null  float64       
dtypes: datetime64[s](1), datetime64[us](1), float64(1), int32(1), str(1)
memory usage: 3.0 MB
None


,datetime,gas_day,hour,site,hourly_site_gen_mw
0,2024-07-23 01:00:00,2024-07-22,1,CGS,39.9
1,2024-07-23 01:00:00,2024-07-22,1,DCS,297.0
2,2024-07-23 01:00:00,2024-07-22,1,GGS,79.4
3,2024-07-23 01:00:00,2024-07-22,1,LCS,186.2
4,2024-07-23 01:00:00,2024-07-22,1,PGS,200.0


In [36]:
#Daily total by site
#######################################################################################################################
daily_site_gen_df = (hourly_df.groupby(["gas_day", "site"], as_index=False)["hourly_mw"].sum().rename(columns={"hourly_mw": "daily_site_gen_mw"}))
daily_site_gen_df['gas_day'] = pd.to_datetime(daily_site_gen_df['gas_day'])

print(daily_site_gen_df.info())
daily_site_gen_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 3655 entries, 0 to 3654
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   gas_day            3655 non-null   datetime64[s]
 1   site               3655 non-null   str          
 2   daily_site_gen_mw  3655 non-null   float64      
dtypes: datetime64[s](1), float64(1), str(1)
memory usage: 85.8 KB
None


,gas_day,site,daily_site_gen_mw
0,2024-07-22,CGS,321.1
1,2024-07-22,DCS,2345.0
2,2024-07-22,GGS,224.8
3,2024-07-22,LCS,1498.1
4,2024-07-22,PGS,1595.0


### Transforming Allegro Data - Burn

Again, I don't yet have access to query Allegro with Python. The raw data has been saved in the checkpoint 1 file in the G drive folder.

These steps were completed earlier

In [37]:
#gas_daily_df = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint1.csv")
gas_daily_df.head()

,begtime,marketarea,energy,site,gas_day
0,2024-07-23,STATELINE,33933.0,PGS,2024-07-23
1,2024-07-23,LONSOME CREEK,42282.0,LCS,2024-07-23
2,2024-07-23,LANARK,13523.0,CGS,2024-07-23
3,2024-07-23,DEER CREEK,46474.0,DCS,2024-07-23
4,2024-07-23,GROTON,11036.0,GGS,2024-07-23


In [35]:
#Adding the columns from the data that we need in the final dataset

#defining market areas as sites
gas_daily_df["marketarea"] = gas_daily_df["marketarea"].str.upper().str.strip()

#removing market areas
gas_daily_df = gas_daily_df[
    ~gas_daily_df["marketarea"].isin(["BISON", "COTTAGE GROVE"])]

site_map = {"DEER CREEK": "DCS", "LANARK": "CGS", "LONSOME CREEK": "LCS",  "STATELINE": "PGS", "GROTON": "GGS", "CULBERTSON": "CGS"}

gas_daily_df["site"] = gas_daily_df["marketarea"].replace(site_map)

gas_daily_df.head()

,begtime,marketarea,energy,site
0,2023-07-22,STATELINE,33476.0,PGS
1,2023-07-22,LANARK,11053.0,CGS
2,2023-07-22,DEER CREEK,13.0,DCS
3,2023-07-22,DEER CREEK,40257.0,DCS
4,2023-07-22,GROTON,9974.0,GGS


In [36]:
# create gas day
gas_daily_df["gas_day"] = pd.to_datetime(gas_daily_df["begtime"])
#gas_daily_df["gas_day"] = (gas_daily_df["gas_day"] - pd.Timedelta(hours=9)).dt.date #Not needed. Begtime is the expected day and contains the date which the NG team refers to as gas_day
gas_daily_df = gas_daily_df.sort_values(by = ['begtime'], ascending=[False])

# Data check
gas_daily_df[gas_daily_df['site']=='DCS'].head(10)

,begtime,marketarea,energy,site,gas_day
6201,2026-07-21,DEER CREEK,54201.0,DCS,2026-07-21
6194,2026-07-20,DEER CREEK,56353.0,DCS,2026-07-20
6190,2026-07-19,DEER CREEK,42669.0,DCS,2026-07-19
6183,2026-07-18,DEER CREEK,49548.0,DCS,2026-07-18
6176,2026-07-17,DEER CREEK,47130.0,DCS,2026-07-17
6168,2026-07-16,DEER CREEK,43164.0,DCS,2026-07-16
6160,2026-07-15,DEER CREEK,50193.0,DCS,2026-07-15
6154,2026-07-14,DEER CREEK,48683.0,DCS,2026-07-14
6147,2026-07-13,DEER CREEK,41660.0,DCS,2026-07-13
6139,2026-07-12,DEER CREEK,26094.0,DCS,2026-07-12


In [37]:
# aggregate on gas_day and renaming energy column to daily_gas_burn 
gas_daily_site = (gas_daily_df.groupby(["gas_day", "site"], as_index=False)["energy"].sum().rename(columns={"energy": "daily_gas_burn"}))
gas_daily_site = gas_daily_site.sort_values(by = ['gas_day'], ascending=[False])
gas_daily_site[gas_daily_site['site']=='DCS'].head(5)

,gas_day,site,daily_gas_burn
5476,2026-07-21,DCS,54201.0
5471,2026-07-20,DCS,56353.0
5466,2026-07-19,DCS,42669.0
5461,2026-07-18,DCS,49548.0
5456,2026-07-17,DCS,47130.0


### Joining Allegro Hourly Generation and Allegro Daily Burn Data

In [39]:
print(hourly_site_gen_df.info())
print(f'\nTotal Generation: {hourly_site_gen_df['hourly_site_gen_mw'].sum()} MW')
print(f'\nDataframe Dimensions: {hourly_site_gen_df.shape}')
#hourly_site_gen_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 87600 entries, 0 to 87599
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            87600 non-null  datetime64[us]
 1   gas_day             87600 non-null  datetime64[s] 
 2   hour                87600 non-null  int32         
 3   site                87600 non-null  str           
 4   hourly_site_gen_mw  87600 non-null  float64       
dtypes: datetime64[s](1), datetime64[us](1), float64(1), int32(1), str(1)
memory usage: 3.0 MB
None

Total Generation: 7675861.152000001 MW

Dataframe Dimensions: (87600, 5)


In [41]:
print(gas_daily_site.info())
print(f'\nDataframe Dimensions: {gas_daily_site.shape}')
#hourly_site_gen_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gas_day         3650 non-null   str    
 1   site            3650 non-null   str    
 2   daily_gas_burn  3650 non-null   float64
dtypes: float64(1), str(2)
memory usage: 85.7 KB
None

Dataframe Dimensions: (3650, 3)


In [43]:
# changeing gas_day to datetime for merge
gas_daily_site['gas_day'] = pd.to_datetime(gas_daily_site['gas_day'])
print(gas_daily_site.info())

<class 'pandas.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   gas_day         3650 non-null   datetime64[us]
 1   site            3650 non-null   str           
 2   daily_gas_burn  3650 non-null   float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 85.7 KB
None


In [42]:
print(daily_site_gen_df.info())
print(f'\nDataframe Dimensions: {daily_site_gen_df.shape}')
#hourly_site_gen_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 3655 entries, 0 to 3654
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   gas_day            3655 non-null   datetime64[s]
 1   site               3655 non-null   str          
 2   daily_site_gen_mw  3655 non-null   float64      
dtypes: datetime64[s](1), float64(1), str(1)
memory usage: 85.8 KB
None

Dataframe Dimensions: (3655, 3)


In [44]:
# Add gas burn data
merged_df = hourly_site_gen_df.merge(gas_daily_site, on=["gas_day", "site"], how="left")
merged_df = merged_df.sort_values(by = ['site', 'gas_day'], ascending=[True, False])

In [47]:
print(f'\nTotal Generation: {merged_df['hourly_site_gen_mw'].sum()} MW')
print(f'Total Burn: {merged_df['daily_gas_burn'].sum()}')
print(f'Dataframe Dimensions: {merged_df.shape}')

merged_df.head(5)


Total Generation: 7675861.152000001 MW
Total Burn: 2009745600.0
Dataframe Dimensions: (87600, 6)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn
87520,2026-07-22 09:00:00,2026-07-22,9,CGS,0.0,0.0
87525,2026-07-22 10:00:00,2026-07-22,10,CGS,0.0,0.0
87530,2026-07-22 11:00:00,2026-07-22,11,CGS,0.0,0.0
87535,2026-07-22 12:00:00,2026-07-22,12,CGS,0.0,0.0
87540,2026-07-22 13:00:00,2026-07-22,13,CGS,0.0,0.0


In [49]:
merged_df = merged_df.merge(daily_site_gen_df, on =["gas_day", "site"], how="left")
merged_df = merged_df.sort_values(by = ['site', 'datetime'], ascending=[True, False])

In [51]:
print(f'\nTotal Generation: {merged_df['hourly_site_gen_mw'].sum()} MW')
print(f'Total Burn: {merged_df['daily_gas_burn'].sum()}')
print(f'Total daily site generation: {merged_df['daily_site_gen_mw'].sum()}')
print(f'Dataframe Dimensions: {merged_df.shape}')

merged_df.to_csv('../output-data/checkpoint11 - merged.csv', index=False)
merged_df.head(5)


Total Generation: 7675861.152 MW
Total Burn: 2009745600.0
Total daily site generation: 184043131.64799997
Dataframe Dimensions: (87600, 7)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
15,2026-07-23 00:00:00,2026-07-22,0,CGS,0.0,0.0,0.0
14,2026-07-22 23:00:00,2026-07-22,23,CGS,0.0,0.0,0.0
13,2026-07-22 22:00:00,2026-07-22,22,CGS,0.0,0.0,0.0
12,2026-07-22 21:00:00,2026-07-22,21,CGS,0.0,0.0,0.0
11,2026-07-22 20:00:00,2026-07-22,20,CGS,0.0,0.0,0.0


In [52]:
# what's best way to avoid div by 0?
# if the daily site generation is 0, can we just divide by 24?
merged_df["hourly_gas_burn"] = np.where(merged_df['daily_site_gen_mw'] != 0, (merged_df["daily_gas_burn"] / merged_df["daily_site_gen_mw"]) * merged_df["hourly_site_gen_mw"], merged_df["daily_gas_burn"]/24)
# merged_df["hourly_gas_burn"] = ((merged_df["daily_gas_burn"] / merged_df["daily_site_gen_mw"]) * merged_df["hourly_site_gen_mw"])

merged_df.to_csv('../output-data/checkpoint12 - with hourly gas burn.csv', index=False)
merged_df[merged_df['daily_site_gen_mw']==0]

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw,hourly_gas_burn
15,2026-07-23 00:00:00,2026-07-22,0,CGS,0.0,0.0,0.0,0.000000
14,2026-07-22 23:00:00,2026-07-22,23,CGS,0.0,0.0,0.0,0.000000
13,2026-07-22 22:00:00,2026-07-22,22,CGS,0.0,0.0,0.0,0.000000
12,2026-07-22 21:00:00,2026-07-22,21,CGS,0.0,0.0,0.0,0.000000
11,2026-07-22 20:00:00,2026-07-22,20,CGS,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...
86084,2024-09-23 13:00:00,2024-09-23,13,PGS,0.0,14.0,0.0,0.583333
86083,2024-09-23 12:00:00,2024-09-23,12,PGS,0.0,14.0,0.0,0.583333
86082,2024-09-23 11:00:00,2024-09-23,11,PGS,0.0,14.0,0.0,0.583333
86081,2024-09-23 10:00:00,2024-09-23,10,PGS,0.0,14.0,0.0,0.583333


In [53]:
# Final output of Allegro data
result_df = merged_df[[ "datetime", "gas_day", "site", "hourly_site_gen_mw",  "daily_site_gen_mw", "daily_gas_burn", "hourly_gas_burn"]]


#math for daily MMBtu per MWh
# Should zero days really be zero or should they be null? My concern would be zero days deflating an average if the intention is to only look at days with generation
# What about days with burn but zero generation? What's the expect gas_per_mw in those instances?
result_df["gas_per_mw"] = (result_df["daily_gas_burn"] / result_df["daily_site_gen_mw"])
result_df.to_csv('../output-data/checkpoint13 - with gas per MW.csv', index=False)

print(f'total generation: {result_df['hourly_site_gen_mw'].sum()}')
result_df.head()

total generation: 7675861.152


,datetime,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw
15,2026-07-23 00:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
14,2026-07-22 23:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
13,2026-07-22 22:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
12,2026-07-22 21:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
11,2026-07-22 20:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN


In [54]:
export_df = result_df.copy()

#makeing a calander date separate from gas day
export_df["date"] = export_df["datetime"].dt.date
export_df.head()


,datetime,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw,date
15,2026-07-23 00:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-23
14,2026-07-22 23:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
13,2026-07-22 22:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
12,2026-07-22 21:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
11,2026-07-22 20:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22


In [55]:
export_df["gas_per_mw"] = (export_df["daily_gas_burn"] / export_df["daily_site_gen_mw"])
export_df.head()

,datetime,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw,date
15,2026-07-23 00:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-23
14,2026-07-22 23:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
13,2026-07-22 22:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
12,2026-07-22 21:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
11,2026-07-22 20:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22


In [56]:
# What is the expected value for instances where there is burn but zero generation?
# I'm not sure changing them to nan is the proper choice...but probably fine for now
export_df["gas_per_mw"] = (export_df["daily_gas_burn"] / export_df["daily_site_gen_mw"]).replace([np.inf, - np.inf], np.nan)
export_df.head()


,datetime,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw,date
15,2026-07-23 00:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-23
14,2026-07-22 23:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
13,2026-07-22 22:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
12,2026-07-22 21:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22
11,2026-07-22 20:00:00,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN,2026-07-22


In [57]:
export_df = export_df[["datetime", "date", "gas_day", "site", "hourly_site_gen_mw", "daily_site_gen_mw", "daily_gas_burn", "hourly_gas_burn", "gas_per_mw"]]
export_df.head()

,datetime,date,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw
15,2026-07-23 00:00:00,2026-07-23,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
14,2026-07-22 23:00:00,2026-07-22,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
13,2026-07-22 22:00:00,2026-07-22,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
12,2026-07-22 21:00:00,2026-07-22,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN
11,2026-07-22 20:00:00,2026-07-22,2026-07-22,CGS,0.0,0.0,0.0,0.0,NaN


In [58]:
# Removed 'gas_day' as the second parameter. Sorting on datetime should implicitly sort by gas_day
# NaNs in daily gas burn are from the data in the csv not being as current as other datasources
    # missing things from July 17th-23rd 2023
export_df = export_df.sort_values(["site", "datetime"])

export_df.to_csv('../output-data/checkpoint14 - Export Dataframe before cleaning.csv', index=False)
export_df.head()


,datetime,date,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw
17512,2024-07-23 01:00:00,2024-07-23,2024-07-22,CGS,39.9,321.1,NaN,NaN,NaN
17513,2024-07-23 02:00:00,2024-07-23,2024-07-22,CGS,40.1,321.1,NaN,NaN,NaN
17514,2024-07-23 03:00:00,2024-07-23,2024-07-22,CGS,39.8,321.1,NaN,NaN,NaN
17515,2024-07-23 04:00:00,2024-07-23,2024-07-22,CGS,40.2,321.1,NaN,NaN,NaN
17516,2024-07-23 05:00:00,2024-07-23,2024-07-22,CGS,40.4,321.1,NaN,NaN,NaN


In [ ]:
print(f'\nTotal Generation: {merged_df['hourly_site_gen_mw'].sum()} MW')
print(f'Total Burn: {merged_df['daily_gas_burn'].sum()}')
print(f'Total daily site generation: {merged_df['daily_site_gen_mw'].sum()}')
print(f'Dataframe Dimensions: {merged_df.shape}')

export_df.shape

(87600, 9)

In [105]:
#I was trying to clean values but I am not sure what the numbers/parameter should be 
# Caleb Question: Is this step needed? Should we be removing records if they are legitimate even if they are outliers
# Caleb Observation: These steps aren't actually used, correct?

clean_df = export_df[(export_df["daily_site_gen_mw"] > -1) & (export_df["gas_per_mw"] > 5) & (export_df["gas_per_mw"] < 20)] #removes 31,200 records that seems 
print(f' previous step removes {export_df.shape[0] - clean_df.shape[0]} records')


 previous step removes 31200 records


In [ ]:
export_df["gas_per_mw"] = pd.to_numeric(export_df["gas_per_mw"], errors="coerce")

In [60]:
#Adding in an export to csv to output the data thus far

#Caleb Observation: We switched back to the merged_df but the previous steps created a df called export_df

result_df = merged_df[["datetime", "gas_day", "site", "daily_site_gen_mw", "hourly_site_gen_mw", "daily_gas_burn", "hourly_gas_burn"]]

result_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\allegro_data.csv", index=False) 

In [61]:
print(f'\nTotal Generation: {result_df['hourly_site_gen_mw'].sum()} MW')
print(f'Total Burn: {result_df['daily_gas_burn'].sum()}')
print(f'Total daily site generation: {result_df['daily_site_gen_mw'].sum()}')
print(f'Dataframe Dimensions: {result_df.shape}')


Total Generation: 7675861.152 MW
Total Burn: 2009745600.0
Total daily site generation: 184043131.64799997
Dataframe Dimensions: (87600, 7)


### Unit Availability Data Import and Load

Right now, these are hardcoded file names. It would be better to get this automated if possible to avoid having to add a new file to the list every month

Possible Issues:
- Hardcoded file list

In [ ]:
#Listing Excel Files to Import
excel_files = [
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 07.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 06.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 05.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 04.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 03.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 02.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 01.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 12.25.xlsx", #There is an issue with this file.
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 11.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 10.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 09.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 08.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 07.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 06.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 05.25 - UPDATE.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 04.25 - UPDATE.xlsx"]


In [107]:
# Listing csv files to import
csv_files = [
    r"G:\Trading\Market Operations\Unit availability\2025\dpm_BEPC_GROUPING_2025030100_2025033123 - March.csv",
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025020100_2025022823 feb.csv",
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025010100_2025013123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024120100_2024123123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024110100_2024113023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024100100_2024103123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024090100_2024093023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024080100_2024083123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024070100_2024073123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024060100_2024063023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024050100_2024053123.csv"]


In [ ]:
def pull_unit_availability(excel_files, csv_files):


    excel_dfs = []
    for file in excel_files:
        if file == "G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 12.25.xlsx"
        df = pd.read_excel(file, sheet_name="Gas HEL Transposed")
        df["source_file"] = file
        excel_dfs.append(df)
        print(f'{file} excel dimensions: {df.shape}') #debugging step
        column_names = df.columns.tolist()

    excel_combined = pd.concat(excel_dfs, ignore_index=True)

    csv_dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df["source_file"] = file
        csv_dfs.append(df)
        print(f'{file} - csv dimensions: {df.shape}') #debugging step

    csv_combined = pd.concat(csv_dfs, ignore_index=True)

    #Concat the csvs and the excel files
    combined_df = pd.concat([excel_combined, csv_combined], ignore_index=True)

    #Making all column names lowercase and remove spaces
    combined_df.columns = (combined_df.columns.str.strip().str.lower())

    #Making datetime column in datetime format
    combined_df["datetime"] = pd.to_datetime( combined_df["datetime"], errors="coerce")

    #Sorting chronologically
    combined_df = (combined_df.sort_values("datetime").reset_index(drop=True))

    return combined_df

In [126]:
# takes a bit of time to run.
df = pull_unit_availability(excel_files, csv_files)

G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 07.26.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 06.26.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 05.26.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 04.26.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 03.26.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 02.26.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 01.26.xlsx excel dimensions: (744, 35)


C:\Users\A105158\AppData\Local\Temp\ipykernel_19516\448884579.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_file"] = file


G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 12.25.xlsx excel dimensions: (16383, 801)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 11.25.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 10.25.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 09.25.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 08.25.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 07.25.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 06.25.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 05.25 - UPDATE.xlsx excel dimensions: (744, 35)
G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 04.25 - UPDATE.xlsx

In [122]:
print(df.shape)
print(df.dtypes)
df.to_csv('../output-data/checkpoint15 - Unit Availability.csv', index=False) 
df.head(5)

(35531, 801)
datetime                       datetime64[us]
cgs1 - high effective limit             int64
dcs1 - high effective limit             int64
ggs1 - high effective limit             int64
ggs2 - high effective limit             int64
                                    ...      
0.761                                 float64
0.762                                 float64
0.763                                 float64
0.764                                 float64
0.765                                 float64
Length: 801, dtype: object


,datetime,cgs1 - high effective limit,dcs1 - high effective limit,ggs1 - high effective limit,ggs2 - high effective limit,lcs1 - high effective limit,lcs2 - high effective limit,lcs3 - high effective limit,lcs4 - high effective limit,lcs5 - high effective limit,...,0.756,0.757,0.758,0.759,0.760,0.761,0.762,0.763,0.764,0.765
0,2024-05-01 00:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-05-01 01:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-05-01 02:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-05-01 03:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-05-01 04:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
